# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading, exploring, and analyzing the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

**Note:** All objects are referenced by their `@id`s as required.

Let's list the record sets and their fields (column `@id`s).

In [ ]:
# Fetch record sets by their @id
record_sets = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]
if not record_sets:
    # If `recordSet` is empty, search for record sets in the metadata graph
    # or print a message. Most current datasets store them in metadata.graph.
    record_sets = []
    all_graph = getattr(metadata, 'graph', getattr(metadata, '@graph', []))
    for entity in all_graph:
        if entity.get('@type') == 'cr:RecordSet':
            record_sets.append(entity['@id'])
    if not record_sets:
        print("No record sets found in metadata.")
    else:
        print(f"Record sets found via metadata.graph: {record_sets}")

# For each record set, show its fields
fields_by_record_set = {}
all_graph = getattr(metadata, 'graph', getattr(metadata, '@graph', []))
for rs_id in record_sets:
    rs_obj = next((x for x in all_graph if x.get('@id') == rs_id), None)
    if rs_obj is not None:
        field_rels = rs_obj.get('cr:field', rs_obj.get('field', []))
        # Normalize to list
        if isinstance(field_rels, dict):
            field_rels = [field_rels]
        elif isinstance(field_rels, list):
            pass
        else:
            field_rels = []
        field_ids = [f['@id'] if isinstance(f, dict) else f for f in field_rels]
        fields_by_record_set[rs_id] = field_ids
    else:
        fields_by_record_set[rs_id] = []

if record_sets:
    for rs_id in record_sets:
        print(f"Record Set @id: {rs_id}")
        print(f"    Fields/columns: [{', '.join(fields_by_record_set[rs_id])}]")
        print()
else:
    print("No record sets available.")

## 3. Data Extraction
Load data from each available record set into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Example: Extract data from all record sets
dataframes = {}

if record_sets:
    for rs_id in record_sets:
        print(f"Loading records from record set {rs_id} ...")
        records = list(dataset.records(record_set=rs_id))
        print(f"  #records loaded: {len(records)}")
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
            print(f"  Columns: {dataframes[rs_id].columns.tolist()}")
            display(dataframes[rs_id].head(3))
        else:
            print(f"  No records in record set {rs_id}.")
else:
    print("No record sets to extract.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section will demonstrate operations like removing outliers and grouping data. 

> **All columns/fields are referenced by their `@id` values.**

Let's select one record set for further analysis. _Update the cell below if you want to analyze a different record set._

In [ ]:
# Pick the main record set for EDA - use the first, or set manually
chosen_record_set = record_sets[0] if record_sets else None
if chosen_record_set is not None and chosen_record_set in dataframes:
    df = dataframes[chosen_record_set]
    print(f"Columns in record set {chosen_record_set}: {df.columns.tolist()}")

    # Try to find a typical numeric field (this logic can be improved for your schema)
    numeric_fields = [col for col in df.columns if df[col].dtype.kind in 'if']
    if not numeric_fields:
        # Try to infer by name
        numeric_fields = [col for col in df.columns if 'coverage' in col.lower() or 'peptide' in col.lower() or 'weight' in col.lower() or 'mw' in col.lower() or 'abundance' in col.lower()]

    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using '{numeric_field}' for numeric analysis.\n")
    else:
        print("No numeric-like field found; please set `numeric_field` manually.")
        numeric_field = None

    if numeric_field is not None:
        # Ensure the numeric field is float for normalization
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0

        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:.2f}: {len(filtered_df)} rows\n")
        display(filtered_df.head(3))

        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() + 1e-8)
        print(f"Normalized values for '{numeric_field}':\n")
        display(filtered_df[[numeric_field, norm_col]].head(3))

        # Try grouping by another field (choose the first string field that's not the id or numeric)
        group_field_candidates = [col for col in df.columns if col != numeric_field and df[col].dtype == object]
        if group_field_candidates:
            group_field = group_field_candidates[0]
            print(f"Grouping by '{group_field}'\n")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            display(grouped_df.head())
        else:
            print("No suitable group/categorical field for grouping.")
    else:
        print("No numeric field selected; cannot proceed with EDA.")
else:
    print("No populated record set DataFrame for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here we plot the distribution of the selected numeric field and, if available, compare across a categorical group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if chosen_record_set is not None and chosen_record_set in dataframes and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field}')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # Group comparison if possible
    if 'group_field' in locals() and group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print('Visualization skipped (no numeric field or record set available).')

## 6. Conclusion
In this notebook, we demonstrated how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library:

- We obtained the Croissant schema and examined the dataset's available record sets and their field (column) `@id`s.
- We loaded the main record set into a pandas DataFrame, explored its structure, filtered and normalized a numeric field, and grouped the data by another attribute.
- We visualized the distribution of a numeric field and, if available, compared distributions across groupings.
- All dataset entities were referenced strictly by their `@id` throughout, ensuring reproducibility and traceability with the metadata.

Proceed to more advanced analysis or modeling steps as needed for your use case!